In [2]:
from typing import List, Tuple

Order = Tuple[int, int]  # (price, volume)


def auction_result(
    bids: List[Order],
    asks: List[Order],
    my_price: int,
    my_volume: int,
    buyback_price: float,
    fee: float = 0.0
):
    """
    Returns:
        clearing_price, filled_quantity, profit
    """

    # --- Split my order ---
    my_is_bid = my_volume > 0
    my_qty = abs(my_volume)

    # Copy books
    bids = bids.copy()
    asks = asks.copy()

    if my_is_bid:
        bids.append((my_price, my_qty))
    else:
        asks.append((my_price, my_qty))

    # --- Find clearing price ---
    prices = sorted({p for p, _ in bids} | {p for p, _ in asks})

    best_price = None
    best_volume = -1

    for p in prices:
        demand = sum(v for price, v in bids if price >= p)
        supply = sum(v for price, v in asks if price <= p)
        traded = min(demand, supply)

        if traded > best_volume or (
            traded == best_volume and (best_price is None or p > best_price)
        ):
            best_volume = traded
            best_price = p

    clearing = best_price

    # --- Compute your fill ---
    filled = 0

    if my_is_bid:
        # Competing demand ahead of you
        better = sum(v for p, v in bids if p > my_price)
        equal_before = sum(v for p, v in bids if p == my_price) - my_qty

        total_demand = sum(v for p, v in bids if p >= clearing)
        total_supply = sum(v for p, v in asks if p <= clearing)
        total_traded = min(total_demand, total_supply)

        remaining = total_traded - better

        if my_price > clearing:
            filled = min(my_qty, remaining)
        elif my_price == clearing:
            remaining -= equal_before
            filled = max(0, min(my_qty, remaining))
        else:
            filled = 0

    else:
        # symmetric for selling
        better = sum(v for p, v in asks if p < my_price)
        equal_before = sum(v for p, v in asks if p == my_price) - my_qty

        total_demand = sum(v for p, v in bids if p >= clearing)
        total_supply = sum(v for p, v in asks if p <= clearing)
        total_traded = min(total_demand, total_supply)

        remaining = total_traded - better

        if my_price < clearing:
            filled = min(my_qty, remaining)
        elif my_price == clearing:
            remaining -= equal_before
            filled = max(0, min(my_qty, remaining))
        else:
            filled = 0

    # --- Profit ---
    if my_is_bid:
        profit_per_unit = buyback_price - clearing - fee
        profit = filled * profit_per_unit
    else:
        # if you sell in auction then buy back (rare case)
        profit_per_unit = clearing - buyback_price - fee
        profit = filled * profit_per_unit

    return clearing, filled, profit

In [ ]:
from typing import List, Tuple

Order = Tuple[int, int]


def find_optimal_order(
    bids: List[Order],
    asks: List[Order],
    buyback_price: float,
    fee: float,
    price_range: List[int],
    max_volume: int,
):
    best = {
        "price": None,
        "volume": 0,
        "profit": float("-inf"),
        "clearing": None,
        "filled": 0,
    }

    for price in price_range:
        for volume in range(1, max_volume + 1): # Note: there's a simple argument why selling will never be optimal
            clearing, filled, profit = auction_result(
                bids, asks, price, volume,
                buyback_price=buyback_price,
                fee=fee
            )

            if profit > best["profit"]:
                best = {
                    "price": price,
                    "volume": volume,
                    "profit": profit,
                    "clearing": clearing,
                    "filled": filled,
                }

    return best

In [4]:
def solve_flax():
    bids = [
        (30, 30000),
        (29, 5000),
        (28, 12000),
        (27, 28000),
    ]

    asks = [
        (28, 40000),
        (31, 20000),
        (32, 20000),
        (33, 30000),
    ]

    return find_optimal_order(
        bids,
        asks,
        buyback_price=30,
        fee=0.0,
        price_range=list(range(25, 35)),
        max_volume=50000,
    )

In [11]:
def solve_mushroom():
    bids = [
        (20, 43000),
        (19, 17000),
        (18, 6000),
        (17, 5000),
        (16, 10000),
        (15, 5000),
        (14, 10000),
        (13, 7000),
    ]

    asks = [
        (12, 20000),
        (13, 25000),
        (14, 35000),
        (15, 6000),
        (16, 5000),
        (17, 0),
        (18, 10000),
        (19, 12000),
    ]

    return find_optimal_order(
        bids,
        asks,
        buyback_price=20,
        fee=0.1,
        price_range=list(range(10, 25)),
        max_volume=100000,
    )

In [ ]:
solve_flax(), solve_mushroom()
# Best orders: buy flax 9999 at 30, buy mushroom 19999 at 17
# Combined PnL: 87995.1

({'price': 30,
  'volume': 9999,
  'profit': 9999.0,
  'clearing': 29,
  'filled': 9999},
 {'price': 17,
  'volume': 19999,
  'profit': 77996.09999999999,
  'clearing': 16,
  'filled': 19999})